# Chapter 8.7 - Densely Connected Networks (DenseNet)

DenseNet takes feature reuse seriously. Instead of adding a block's new features to the old representation, it concatenates them. Every later layer in a dense block receives all earlier feature maps as input.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs. Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

## You are done when you can

- explain how DenseNet differs from ResNet
- implement a dense block with channel growth
- compute the output channels from growth rate and block depth
- use transition layers to compress channels and downsample space
- debug concatenating along the wrong dimension


In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current


## 8.7.0 The Problem This Notebook Solves

ResNet combines old and new information by addition:

```text
new representation = old representation + update
```

DenseNet combines old and new information by concatenation:

```text
new representation = concatenate(old features, newly computed features)
```

Addition keeps channel count the same. Concatenation grows channel count. DenseNet's growth rate controls how many new channels each layer adds.


## 8.7.1 Residual Addition and Dense Concatenation Preserve Information Differently

Addition merges two tensors into the same channel slots. Concatenation keeps both tensors as separate channel slices.

This is not automatically better or worse. It is a different inductive bias:

- ResNet says: learn an update to the current representation.
- DenseNet says: keep earlier features directly available to later layers.


In [ ]:
old = torch.randn(2, 4, 8, 8)
update = torch.randn(2, 4, 8, 8)

residual_style = old + update
dense_style = torch.cat([old, update], dim=1)

print("residual shape:", shape(residual_style))
print("dense shape:", shape(dense_style))

assert shape(residual_style) == (2, 4, 8, 8)
assert shape(dense_style) == (2, 8, 8, 8)


## 8.7.2 A Dense Block Grows Channels by the Growth Rate

Each layer in a dense block receives all previous features and produces `growth_rate` new channels. Those new channels are concatenated onto the running representation.

If the input has `C` channels, the block has `L` layers, and the growth rate is `G`, then:

```text
output channels = C + L * G
```


In [ ]:
def conv_block(in_channels, growth_rate):
    return nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(),
        nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1),
    )


class DenseBlock(nn.Module):
    def __init__(self, in_channels, num_convs, growth_rate):
        super().__init__()
        self.blocks = nn.ModuleList()
        channels = in_channels
        for _ in range(num_convs):
            self.blocks.append(conv_block(channels, growth_rate))
            channels += growth_rate
        self.out_channels = channels

    def forward(self, X):
        features = X
        self.channel_trace = [features.shape[1]]
        for block in self.blocks:
            new_features = block(features)
            features = torch.cat([features, new_features], dim=1)
            self.channel_trace.append(features.shape[1])
        return features


dense_block = DenseBlock(6, num_convs=3, growth_rate=4)
Y = dense_block(torch.randn(2, 6, 8, 8))

print("channel trace:", dense_block.channel_trace)
assert shape(Y) == (2, 18, 8, 8)
assert dense_block.out_channels == 18


## 8.7.3 Transition Layers Compress Channels and Downsample

Dense blocks grow channel count. Without a control mechanism, the network becomes increasingly wide. A transition layer usually performs:

```text
BatchNorm -> ReLU -> 1 by 1 convolution -> average pooling
```

The 1 by 1 convolution compresses channels. Average pooling reduces spatial size.


In [ ]:
def transition_block(in_channels, out_channels):
    return nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(),
        nn.Conv2d(in_channels, out_channels, kernel_size=1),
        nn.AvgPool2d(kernel_size=2, stride=2),
    )


transition = transition_block(18, 10)
Y2 = transition(Y)

print("after transition:", shape(Y2))
assert shape(Y2) == (2, 10, 4, 4)


## 8.7.4 Build a Tiny DenseNet

The DenseNet pattern is:

```text
stem -> dense block -> transition -> dense block -> global average pool -> classifier
```

The builder must track channel count carefully because dense blocks change it by concatenation.


In [ ]:
class TinyDenseNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 6, kernel_size=3, padding=1))
        self.block1 = DenseBlock(6, num_convs=2, growth_rate=4)
        self.trans1 = transition_block(self.block1.out_channels, 10)
        self.block2 = DenseBlock(10, num_convs=2, growth_rate=4)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
                                  nn.Linear(self.block2.out_channels, num_classes))

    def forward(self, X):
        X = self.stem(X)
        X = self.block1(X)
        X = self.trans1(X)
        X = self.block2(X)
        return self.head(X)


model = TinyDenseNet()
logits = model(torch.randn(2, 1, 16, 16))

print("block1 out channels:", model.block1.out_channels)
print("block2 out channels:", model.block2.out_channels)
print("logits:", shape(logits))

assert shape(logits) == (2, 10)


## 8.7.5 Break It Deliberately: Concatenate Along the Wrong Dimension

DenseNet growth is channel growth. If you concatenate along height or width, the code might produce a tensor, but it is not a dense feature stack.

The theory-level mistake is confusing "put tensors together" with "add feature channels for later layers".


In [ ]:
X = torch.randn(2, 6, 8, 8)
new = torch.randn(2, 4, 8, 8)
new_same_channels = torch.randn(2, 6, 8, 8)

correct = torch.cat([X, new], dim=1)
wrong = torch.cat([X, new_same_channels], dim=2)

print("correct:", shape(correct))
print("wrong:", shape(wrong))

try:
    assert wrong.shape[1] == X.shape[1] + new.shape[1]
except AssertionError:
    print("The tensor was concatenated, but channels did not grow.")


## 8.7 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. How does DenseNet combine old and new features differently from ResNet?
2. What is the growth rate?
3. Why does a dense block need careful channel bookkeeping?
4. What does a transition layer do?
5. Why is concatenating along height not the same as DenseNet feature reuse?
